# AgriResolve AI — exploratory data analysis

Reads `data/cleaned` and `data/derived` produced by `python -m src.utils.pipeline`.

**All observations are synthetic.** Every chart filters or colours on `data_type` / `is_synthetic`
so synthetic series can never be mistaken for government data.

Run this notebook from the project root.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px
from src.config.settings import get_config, project_path

cfg = get_config()
cleaned = project_path('data', 'cleaned')
derived = project_path('data', 'derived')

villages = pd.read_csv(cleaned / 'villages.csv')
rainfall = pd.read_csv(cleaned / 'rainfall_monthly.csv')
ndvi = pd.read_csv(cleaned / 'ndvi_monthly.csv')
complaints = pd.read_csv(cleaned / 'farmer_complaints.csv', parse_dates=['reported_date'])
features = pd.read_csv(derived / 'village_features.csv')
priority = pd.read_csv(derived / 'priority_scores.csv')

print('reference date:', cfg.reference_date, '| synthetic rows only:', bool(rainfall.is_synthetic.all()))
villages[['village_id', 'village_name', 'cultivated_area_ha', 'source_status']]

## 1. Rainfall versus each village's own baseline

Absolute millimetres are not comparable across villages; deviation from the village baseline is.

In [ ]:
fig = px.bar(
    features.sort_values('rainfall_deviation'),
    x='rainfall_deviation', y='village_name', orientation='h',
    labels={'rainfall_deviation': 'Season rainfall deviation (fraction)', 'village_name': ''},
    title='Kharif 2026 rainfall deviation vs 2019-2025 baseline (synthetic)',
    text=features.sort_values('rainfall_deviation')['rainfall_deviation'].round(3),
)
fig.update_layout(height=360)
fig.show()

In [ ]:
rainfall['period'] = pd.to_datetime(dict(year=rainfall.year, month=rainfall.month, day=1))
fig = px.line(
    rainfall[rainfall.village_id.isin(['V01', 'V02', 'V06'])],
    x='period', y='rainfall_mm', color='village_id',
    title='Monthly rainfall, 2019-2026 (synthetic): drought vs flood vs normal',
)
fig.add_vrect(x0='2026-06-01', x1='2026-10-31', fillcolor='grey', opacity=0.15, annotation_text='Kharif 2026')
fig.update_layout(height=380)
fig.show()

## 2. NDVI trend and the cloud-gap problem

Missing months are kept as gaps (not imputed); Gavadagere is deliberately cloud-limited.

In [ ]:
ndvi['period'] = pd.to_datetime(dict(year=ndvi.year, month=ndvi.month, day=1))
fig = px.line(
    ndvi, x='period', y='ndvi_mean', color='village_id',
    title='Monthly NDVI (synthetic) — gaps are months with no reliable retrieval',
)
fig.update_layout(height=380)
fig.show()

features[['village_name', 'completeness', 'cloud_cover_avg', 'ndvi_recency_days', 'reliability_score']].round(3)

## 3. Complaints: category mix and concentration

Concentration matters more than the total: repeats in one village drive severity.

In [ ]:
by_village = complaints.groupby(['village_id', 'category']).size().reset_index(name='complaints')
fig = px.bar(
    by_village, x='village_id', y='complaints', color='category', barmode='stack',
    title='Complaints per village and category (synthetic, last ~45 days)',
)
fig.update_layout(height=360)
fig.show()

complaints.groupby('category').size().sort_values(ascending=False)

## 4. Inspection gaps and data completeness

Which villages have not been looked at recently, and can we even judge them from the desk?

In [ ]:
fig = px.scatter(
    features, x='days_since_inspection', y='reliability_score',
    size='cultivated_area_ha', color='rainfall_deviation', text='village_name',
    labels={'days_since_inspection': 'Days since last inspection', 'reliability_score': 'Data reliability'},
    title='Neglect vs data reliability (bubble size = cultivated area)',
)
fig.add_hline(y=0.6, line_dash='dash', annotation_text='low-reliability cutoff')
fig.update_traces(textposition='top center')
fig.update_layout(height=420)
fig.show()

## 5. Resulting priority

The engine output, for comparison with the observations above.

In [ ]:
fig = px.bar(
    priority.sort_values('priority_score'), x='priority_score', y='village_name',
    orientation='h', color='band', text='priority_score',
    title='Combined priority score (synthetic data, weights in src/config/priority_weights.yaml)',
)
fig.update_layout(height=360)
fig.show()
priority[['rank', 'village_name', 'priority_score', 'band', 'overlap_detected', 'dedupe_rule']]

## Findings

See `docs/eda_findings.md` for the written interpretation, the threshold choices it justifies and the
caveat that all correlations here were constructed by the generator.